In [75]:
import pandas as pd             # For handling data (DataFrame)
import seaborn as sns           # For visualizations and Planets dataset
import numpy as np              # For numerical operations
# import matplotlib.pyplot as plt # For plots
# import math

# from pandas import plotting

# import statsmodels.formula.api as smf
# from sklearn.linear_model import LinearRegression
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_squared_error, r2_score

In [76]:
crash_more_grouping_df = pd.read_csv(
    "https://raw.githubusercontent.com/Alissa-Ouspen/data201_alissa/main/final/crash_data_grouped_1.csv")


    #usecols = lambda x: x not in ["Second_Harmful_Event", "Driverless_Vehicle", "Veh_Going_Dir", "Route_Type",])

###crash_data_grouped_1.csv
--------------
Collision_Type, First_Harmful_Event, and Driver_Distraction remain,  
although probably won't use for current analysis.

###Working Here, Below


In [77]:
crash_more_grouping_df.head()


,Vehicle_Num,Agency_Name,ACRS_Report_Type,Crash_Date_Time,Related_Non_Motorist,Collision_Type,Weather,Surface_Condition,Ambient_Light,Traffic_Control,...,Parked_Vehicle,Vehicle_Year,Hit_Run,Lane_Type,At_Fault,First_Harmful_Event,Junction,Intersection_Type,Road_Alignment,Road_Condition
0,0,Montgomery County Police,Injury Crash,5/5/26 22:43,NaN,Front to Front,Clear,Dry,Dark - Lighted,No Controls,...,No,2018.0,No,Lane 2,DRIVER,MOTOR VEHICLE IN TRANSPORT,NaN,NaN,Straight,No Defects
1,1,Montgomery County Police,Injury Crash,5/5/26 22:43,NaN,Front to Front,Clear,Dry,Dark - Lighted,No Controls,...,No,2026.0,No,Lane 2,DRIVER,MOTOR VEHICLE IN TRANSPORT,NaN,NaN,Straight,No Defects
2,2,Rockville Police Dept,Property Damage Crash,5/5/26 21:43,NaN,Front to Rear,Clear,Dry,Dark - Lighted,Flashing Traffic Control Signal,...,No,2023.0,No,Lane 2,NaN,MOTOR VEHICLE IN TRANSPORT,ACCELERATION/DECELERATION LANE,NaN,Straight,No Defects
3,3,Rockville Police Dept,Property Damage Crash,5/5/26 21:43,NaN,Front to Rear,Clear,Dry,Dark - Lighted,Flashing Traffic Control Signal,...,No,2005.0,No,Lane 2,NaN,MOTOR VEHICLE IN TRANSPORT,ACCELERATION/DECELERATION LANE,NaN,Straight,No Defects
4,4,Montgomery County Police,Property Damage Crash,5/5/26 21:30,NaN,Angle,Clear,Dry,Dark - Lighted,Lane Use Control Signal,...,No,2025.0,No,Lane 1,DRIVER,MOTOR VEHICLE IN TRANSPORT,CROSSOVER-RELATED,NaN,Straight,No Defects


In [78]:
crash_more_grouping_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214273 entries, 0 to 214272
Data columns (total 30 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Vehicle_Num                 214273 non-null  int64  
 1   Agency_Name                 214273 non-null  object 
 2   ACRS_Report_Type            214273 non-null  object 
 3   Crash_Date_Time             214273 non-null  object 
 4   Related_Non_Motorist        6767 non-null    object 
 5   Collision_Type              191162 non-null  object 
 6   Weather                     199566 non-null  object 
 7   Surface_Condition           189141 non-null  object 
 8   Ambient_Light               211494 non-null  object 
 9   Traffic_Control             182332 non-null  object 
 10  Driver_Substance_Use        170904 non-null  object 
 11  Non_Motorist_Substance_Use  5654 non-null    object 
 12  Driver_At_Fault             209591 non-null  object 
 13  Injury_Severit

-----------
###Junction
Junction is sufficiently grouped to be used in analysis

"OTHER" includes
SHARED-USE PATH OR TRAIL,
RAILWAY GRADE CROSSING,
ALLEY

In [79]:
# Clean Junction: "Non-Junction" = "NON INTERSECTION" = "UNKNOWN" = nan
# Convert to uppercase for consistency
crash_more_grouping_df["Junction"] = crash_more_grouping_df["Junction"].astype(str).str.upper()

values_to_replace_junction = ["NON-JUNCTION", "NON INTERSECTION", "UNKNOWN", "NAN", "NA"]
crash_more_grouping_df["Junction"] = crash_more_grouping_df["Junction"].replace(values_to_replace_junction, np.nan)

crash_more_grouping_df["Junction"] = crash_more_grouping_df["Junction"].replace({
    "INTERSECTION" : "INTERSECTION RELATED",
    "INTERSECTION OR RELATED" : "INTERSECTION RELATED",
    "CROSSOVER-RELATED" : "CROSSOVER RELATED",
    "RESIDENTIAL DRIVEWAY" : "DRIVEWAY - RESIDENTIAL",
    "COMMERCIAL DRIVEWAY" : "DRIVEWAY - COMMERCIAL",
    "DRIVEWAY ACCESS OR RELATED" : "DRIVEWAY - OTHER (ACCESS/RELATED)",
    "OTHER DRIVEWAY" : "DRIVEWAY - OTHER (ACCESS/RELATED)",
    "OTHER LOCATION NOT LISTED ABOVE WITHIN AN INTERCHANGE AREA (MEDIAN,\nSHOULDER AND ROADSIDE)" : "INTERCHANGE RELATED INCL MEDIAN, SHOULDER, AND ROADSIDE",
    "INTERCHANGE RELATED" : "INTERCHANGE RELATED INCL MEDIAN, SHOULDER, AND ROADSIDE",
    "SHARED-USE PATH OR TRAIL" : "OTHER",
    "RAILWAY GRADE CROSSING" : "OTHER",
    "ALLEY" : "OTHER"

})

print("Unique values after cleaning Junction:")
print(crash_more_grouping_df["Junction"].unique())
print("\nValue counts for")
print(crash_more_grouping_df["Junction"].value_counts(dropna=False).sort_index())
print("\nTotal number of groups:")
print(crash_more_grouping_df["Junction"].nunique())

Unique values after cleaning Junction:
[nan 'ACCELERATION/DECELERATION LANE' 'CROSSOVER RELATED'
 'INTERSECTION RELATED' 'ENTRANCE/EXIT RAMP OR RELATED' 'THROUGH ROADWAY'
 'DRIVEWAY - OTHER (ACCESS/RELATED)'
 'INTERCHANGE RELATED INCL MEDIAN, SHOULDER, AND ROADSIDE' 'OTHER'
 'DRIVEWAY - COMMERCIAL' 'DRIVEWAY - RESIDENTIAL']

Value counts for
Junction
ACCELERATION/DECELERATION LANE                               1127
CROSSOVER RELATED                                            1517
DRIVEWAY - COMMERCIAL                                        2199
DRIVEWAY - OTHER (ACCESS/RELATED)                            1585
DRIVEWAY - RESIDENTIAL                                        816
ENTRANCE/EXIT RAMP OR RELATED                                 880
INTERCHANGE RELATED INCL MEDIAN, SHOULDER, AND ROADSIDE      1951
INTERSECTION RELATED                                        97235
OTHER                                                          77
THROUGH ROADWAY                                      

-------------
###Collision_Type:
Still a lot of categories, but I don't think I can combine categories at this time.  No redundancies apparent.

In [80]:
# Clean Collision_Type: 'other' = 'unknown' = nan
values_to_replace_ct = ["other", "unknown", "OTHER", "UNKNOWN"]
crash_more_grouping_df["Collision_Type"] = crash_more_grouping_df["Collision_Type"].astype(str).replace(values_to_replace_ct, np.nan, regex=True)

print("Unique values after cleaning Collision_Type:")
print(crash_more_grouping_df["Collision_Type"].unique())
print("\nValue counts for Collision_Type:")
print(crash_more_grouping_df["Collision_Type"].value_counts(dropna=False).sort_index())

Unique values after cleaning Collision_Type:
['Front to Front' 'Front to Rear' 'Angle' 'Single Vehicle'
 'Sideswipe, Opposite Direction' 'nan' 'Rear To Side' 'Rear To Rear'
 'Sideswipe, Same Direction' 'OPPOSITE DIRECTION SIDESWIPE'
 'HEAD ON LEFT TURN' 'STRAIGHT MOVEMENT ANGLE' 'SAME DIR REAR END'
 'HEAD ON' 'SAME DIRECTION SIDESWIPE' 'SAME DIRECTION RIGHT TURN'
 'SAME DIRECTION LEFT TURN' 'ANGLE MEETS LEFT TURN'
 'SAME DIR BOTH LEFT TURN' 'SAME DIR REND LEFT TURN'
 'ANGLE MEETS RIGHT TURN' 'OPPOSITE DIR BOTH LEFT TURN'
 'ANGLE MEETS LEFT HEAD ON' 'SAME DIR REND RIGHT TURN']

Value counts for Collision_Type:
Collision_Type
ANGLE MEETS LEFT HEAD ON           700
ANGLE MEETS LEFT TURN             2033
ANGLE MEETS RIGHT TURN            1204
Angle                             8897
Front to Front                    2352
Front to Rear                    12494
HEAD ON                           3786
HEAD ON LEFT TURN                12926
OPPOSITE DIR BOTH LEFT TURN        322
OPPOSITE DIRECTIO

In [81]:
rear_end_collisions = [
    'SAME DIR REAR END',
    'Front to Rear',
    'Rear To Rear',
    'SAME DIR REND LEFT TURN',
    'SAME DIR REND RIGHT TURN'
]

sideswipe_collisions = [
    'Sideswipe, Same Direction',
    'SAME DIRECTION SIDESWIPE',
    'Sideswipe, Opposite Direction',
    'OPPOSITE DIRECTION SIDESWIPE'
]

angle_collisions = [
    'STRAIGHT MOVEMENT ANGLE',
    'ANGLE MEETS LEFT TURN',
    'ANGLE MEETS RIGHT TURN',
    'ANGLE MEETS LEFT HEAD ON',
    'Angle'
]

head_on_collisions = [
    'HEAD ON LEFT TURN',
    'HEAD ON',
    'Front to Front',
    'OPPOSITE DIR BOTH LEFT TURN'
]

other_single_vehicle = [
    'Single Vehicle',
    'SAME DIR BOTH LEFT TURN',
    'SAME DIRECTION LEFT TURN',
    'SAME DIRECTION RIGHT TURN'
]

# Apply the groupings
crash_more_grouping_df['Collision_Type'] = crash_more_grouping_df['Collision_Type'].replace(rear_end_collisions, 'REAR END COLLISIONS')
crash_more_grouping_df['Collision_Type'] = crash_more_grouping_df['Collision_Type'].replace(sideswipe_collisions, 'SIDESWIPE COLLISIONS')
crash_more_grouping_df['Collision_Type'] = crash_more_grouping_df['Collision_Type'].replace(angle_collisions, 'ANGLE COLLISIONS')
crash_more_grouping_df['Collision_Type'] = crash_more_grouping_df['Collision_Type'].replace(head_on_collisions, 'HEAD ON COLLISIONS')
crash_more_grouping_df['Collision_Type'] = crash_more_grouping_df['Collision_Type'].replace(other_single_vehicle, 'OTHER / SINGLE VEHICLE')

print("\nValue counts for Collision_Type after grouping:")
print(crash_more_grouping_df['Collision_Type'].value_counts(dropna=False).sort_index())
print("\nTotal number of groups after grouping:")
print(crash_more_grouping_df['Collision_Type'].nunique())


Value counts for Collision_Type after grouping:
Collision_Type
ANGLE COLLISIONS          43176
HEAD ON COLLISIONS        19386
OTHER / SINGLE VEHICLE    29084
REAR END COLLISIONS       70221
Rear To Side               1909
SIDESWIPE COLLISIONS      27386
nan                       23111
Name: count, dtype: int64

Total number of groups after grouping:
7


###First_Harmful_Event
Not ready for analysis

In [82]:


print("\nValue counts for First_Harmful_Event:")
print(crash_more_grouping_df["First_Harmful_Event"].value_counts(dropna=False).sort_index())
print("\nTotal number of groups:")
print(crash_more_grouping_df["First_Harmful_Event"].nunique())


Value counts for First_Harmful_Event:
First_Harmful_Event
ANIMAL                                                                             964
ANIMAL (LIVE)                                                                      205
BACKING                                                                            151
BICYCLE                                                                            942
BRIDGE OVERHEAD STRUCTURE                                                            3
BRIDGE PIER OR SUPPORT                                                               2
BRIDGE RAIL                                                                         10
CABLE BARRIER                                                                        2
CARGO/EQUIPMENT LOSS OR SHIFT                                                        2
CONCRETE TRAFFIC BARRIER                                                            71
CONSTRUCTION EQUIPMENT                                                 

In [83]:
# Clean First_Harmful_Event: Remove common 'unknown' / 'other' patterns
# Display unique values to identify cleaning needs, then apply standard replacements
# Convert the column to string type at the beginning to handle mixed types consistently
crash_more_grouping_df["First_Harmful_Event"] = crash_more_grouping_df["First_Harmful_Event"].astype(str)

fixed_pole_objects = [
    'TREE (STANDING)',
    'OTHER POST, POLE, OR SUPPORT',
    'UTILITY POLE/LIGHT SUPPORT',
    'TRAFFIC SIGN SUPPORT',
    'TRAFFIC SIGNAL SUPPORT',
    'MAILBOX',
    'BRIDGE PIER OR SUPPORT'
]

traffic_barrier_devices = [
    'GUARDRAIL FACE',
    'CONCRETE TRAFFIC BARRIER',
    'OTHER TRAFFIC BARRIER',
    'CABLE BARRIER',
    'GUARDRAIL END TERMINAL',
    'BRIDGE RAIL',
    'IMPACT ATTENUATOR/CRASH CUSHION',
    'FENCE'
]

other_fixed_objects = [
    'CULVERT',
    'BRIDGE OVERHEAD STRUCTURE',
    'CURB',
    "FIXED OBJECT"
]

animal_group = [
    'ANIMAL (LIVE)',
    'ANIMAL'
]

overturn_group = [
    'OVERTURN/ROLLOVER',
    'OVERTURN'
]

non_collision_events = [
    'FELL JUMPED FROM MOTOR VEHICLE',
    'EXPLOSION OR FIRE'
]

cargo_falling_object_related = [
    'SPILLED CARGO',
    'CARGO/EQUIPMENT LOSS OR SHIFT',
    'STRUCK BY FALLING, SHIFTING CARGO OR ANYTHING SET IN MOTION BY MOTOR VEHICLE',
    'THROWN OR FALLING OBJECT'
]

went_off_road = [
    'DITCH',
    'EMBANKMENT',
    'OFF ROAD',
    'IMMERSION'
]

cargo_truck_issues = [
    'JACKKNIFE',
    'UNITS SEPARATED',
    'DOWNHILL RUNAWAY'
]

operational_issues = [
    'BACKING',
    'U-TURN'
]

other_group = [
    'STRIKES OBJECT AT REST FROM MOTOR VEHICLE IN TRANSPORT',
    'RAILWAY TRAIN',
    'CONSTRUCTION EQUIPMENT',
    'RAILWAY VEHICLE (TRAIN, ENGINE)'
]

# Apply all specific groupings to the temporary series
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(fixed_pole_objects, 'COLLISION WITH FIXED POLE OBJECT')
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(traffic_barrier_devices, 'COLLISION WITH TRAFFIC BARRIER / PROTECTIVE DEVICE')
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(other_fixed_objects, 'COLLISION WITH UNSPECIFIED FIXED OBJECT')
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(animal_group, 'ANIMAL RELATED')
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(overturn_group, 'OVERTURN/ROLLOVER EVENT')
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(non_collision_events, 'NON-COLLISION EVENT')
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(cargo_falling_object_related, 'CARGO OR FALLING OBJECT RELATED')
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(went_off_road, 'WENT OFF ROAD')
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(cargo_truck_issues, 'CARGO TRUCK ISSUES')
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(operational_issues, 'OPERATIONAL ISSUES')
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(other_group, 'OTHER EVENT')


values_to_replace = ["unknown", "other", "UNKNOWN", 'nan']
crash_more_grouping_df['First_Harmful_Event'] = crash_more_grouping_df['First_Harmful_Event'].replace(values_to_replace, np.nan, regex=False)

print("\nValue counts for First_Harmful_Event after grouping:")
print(crash_more_grouping_df['First_Harmful_Event'].value_counts(dropna=False).sort_index())
print("\nTotal number of groups after grouping:")
print(crash_more_grouping_df['First_Harmful_Event'].nunique())


Value counts for First_Harmful_Event after grouping:
First_Harmful_Event
ANIMAL RELATED                                          1169
BICYCLE                                                  942
CARGO OR FALLING OBJECT RELATED                          139
CARGO TRUCK ISSUES                                        32
COLLISION WITH FIXED POLE OBJECT                        1114
COLLISION WITH TRAFFIC BARRIER / PROTECTIVE DEVICE       395
COLLISION WITH UNSPECIFIED FIXED OBJECT                13106
MOTOR VEHICLE IN TRANSPORT                             33945
NON-COLLISION EVENT                                       80
OPERATIONAL ISSUES                                       176
OTHER CONVEYANCE                                         132
OTHER EVENT                                               40
OTHER FIXED OBJECT (WALL, BUILDING, TUNNEL, ETC.)        384
OTHER NON COLLISION                                      159
OTHER NON-COLLISION                                       73
OTHER NON-F

--------------------
-----------
##First_Harmful_Event
-------------
COLLISION WITH FIXED POLE OBJECTS:

TREE (STANDING)  
OTHER POST, POLE, OR SUPPORT  
UTILITY POLE/LIGHT SUPPORT  
TRAFFIC SIGN SUPPORT  
TRAFFIC SIGNAL SUPPORT  
MAILBOX  
BRIDGE PIER OR SUPPORT  


COLLISION WITH TRAFFIC BARRIER / PROTECTIVE DEVICE:
GUARDRAIL FACE, CONCRETE TRAFFIC BARRIER, OTHER TRAFFIC BARRIER, CABLE BARRIER, GUARDRAIL END TERMINAL, BRIDGE RAIL, IMPACT ATTENUATOR/CRASH CUSHION, FENCE  


COLLISION WITH OTHER FIXED OBJECT:
CULVERT,
BRIDGE OVERHEAD STRUCTURE,
CURB


#### Analysis and Suggested Groupings for 'First_Harmful_Event':

Based on the unique values in the `First_Harmful_Event` column, here are some categories that appear to be similar or related and could potentially be combined into larger groups:

**1. Collisions with Animals:**
*   `ANIMAL (LIVE)` and `ANIMAL` could be combined into a general `ANIMAL` category.

**2. Collisions with Fixed Objects (Roadside Furniture, Infrastructure):**
*   `FENCE`
*   `CURB`
*   `TREE (STANDING)`
*   `OTHER POST, POLE, OR SUPPORT`
*   `UTILITY POLE/LIGHT SUPPORT`
*   `TRAFFIC SIGN SUPPORT`
*   `TRAFFIC SIGNAL SUPPORT`
*   `MAILBOX`
*   `BRIDGE PIER OR SUPPORT`
*   `GUARDRAIL FACE`, `CONCRETE TRAFFIC BARRIER`, `OTHER TRAFFIC BARRIER`, `CABLE BARRIER`, `GUARDRAIL END TERMINAL`, `BRIDGE RAIL`, `IMPACT ATTENUATOR/CRASH CUSHION` could be grouped under `TRAFFIC BARRIER / PROTECTIVE DEVICES`.
*   `OTHER FIXED OBJECT (WALL, BUILDING, TUNNEL, ETC.)` and `FIXED OBJECT` could be combined into a `GENERAL FIXED OBJECT` category.
*   `CULVERT`
*   `BRIDGE OVERHEAD STRUCTURE`

**3. Overturn/Rollover Events:**
*   `OVERTURN/ROLLOVER` and `OVERTURN` are essentially the same and can be combined.

**4. Non-Collision Events (excluding overturns):**
*   `OTHER NON-COLLISION` and `OTHER NON COLLISION` can be combined.
*   `FELL JUMPED FROM MOTOR VEHICLE`
*   `EXPLOSION OR FIRE`
*   `IMMERSION`

**5. Collisions with Other Vehicles/Non-Motorists:**
*   `MOTOR VEHICLE IN TRANSPORT`
*   `PARKED VEHICLE`
*   `PEDALCYCLE` (and `BICYCLE` if considered distinct enough, otherwise combine with pedalcycle)
*   `PEDESTRIAN`
*   `OTHER NON-MOTORIST`
*   `OTHER VEHICLE`
*   `RAILWAY VEHICLE (TRAIN, ENGINE)` and `RAILWAY TRAIN`
*   `OTHER PEDALCYCLE`

**6. Cargo/Load Related Events:**
*   `SPILLED CARGO` and `CARGO/EQUIPMENT LOSS OR SHIFT` could be combined into `CARGO RELATED`.
*   `THROWN OR FALLING OBJECT`
*   `STRUCK BY FALLING, SHIFTING CARGO OR ANYTHING SET IN MOTION BY MOTOR VEHICLE`

**7. Roadway-related Features:**
*   `DITCH`
*   `EMBANKMENT`
*   `OFF ROAD`

**8. Vehicle Malfunction/Operational Issues:**
*   `JACKKNIFE`
*   `UNITS SEPARATED`
*   `DOWNHILL RUNAWAY`
*   `BACKING`
*   `U-TURN`

**9. Other/Miscellaneous:**
*   `CONSTRUCTION EQUIPMENT`
*   `OTHER CONVEYANCE`
*   `nan` (Represents missing values, which are usually kept separate or handled through imputation).

These suggestions aim to reduce the cardinality of the 'First_Harmful_Event' column by grouping semantically similar events, which can be beneficial for analysis and modeling.

-----------
###Road_Alignment


In [84]:
# Rename the column first
crash_more_grouping_df = crash_more_grouping_df.rename(columns={"Road Alignment": "Road_Alignment"})

# Preserve original NaN values
original_nan_mask = crash_more_grouping_df["Road_Alignment"].isna()

# Convert to string and title case for standardization, but only for non-NaN values
# This will turn np.nan into the string 'Nan'
crash_more_grouping_df["Road_Alignment"] = crash_more_grouping_df["Road_Alignment"].astype(str).str.title()

# Standardize 'Straight' values
straight_patterns = ["Straight", "Straight, Straight"]
crash_more_grouping_df["Road_Alignment"] = crash_more_grouping_df["Road_Alignment"].replace(straight_patterns, "Straight")

# Convert all variations of 'Curve' to 'Curved'
curve_patterns = [
    "Curve Left", "Curve Left, Straight", "Curve Right",
    "Curve Left, Curve Right", "Curve Right, Straight", "Curve Left, Curve Right, Straight"
]
crash_more_grouping_df["Road_Alignment"] = crash_more_grouping_df["Road_Alignment"].replace(curve_patterns, "Curved")

# Now, convert the string 'Nan' (which came from original np.nan) back to actual np.nan
crash_more_grouping_df["Road_Alignment"] = crash_more_grouping_df["Road_Alignment"].replace("Nan", np.nan)

# Ensure that any values that were originally NaN are still NaN after all processing
crash_more_grouping_df.loc[original_nan_mask, "Road_Alignment"] = np.nan

print("Unique values after cleaning Road_Alignment:")
print(crash_more_grouping_df["Road_Alignment"].unique())
print("\nValue counts for Road_Alignment:")
print(crash_more_grouping_df["Road_Alignment"].value_counts(dropna=False))

Unique values after cleaning Road_Alignment:
['Straight' nan 'Curved']

Value counts for Road_Alignment:
Road_Alignment
Straight    173838
NaN          21660
Curved       18775
Name: count, dtype: int64


In [85]:
# Clean Vehicle_Damage: caseblind, "unkonwn" = "other" = nan
values_to_replace_vd = ["unkonwn", "other", "UNKNOWN", "OTHER", "nan"]
crash_more_grouping_df["Vehicle_Damage"] = crash_more_grouping_df["Vehicle_Damage"].astype(str).replace(values_to_replace_vd, np.nan, regex=True)
crash_more_grouping_df["Vehicle_Damage"] = crash_more_grouping_df["Vehicle_Damage"].replace("DESTROYED", "Destroyed")

print("\nValue counts for Vehicle_Damage:")
print(crash_more_grouping_df["Vehicle_Damage"].value_counts(dropna=False))


Value counts for Vehicle_Damage:
Vehicle_Damage
Disabling               79964
Superficial             55186
Functional              54644
Destroyed                7610
NaN                      7038
No Damage                6691
Vehicle Not at Scene     3140
Name: count, dtype: int64


In [86]:
import numpy as np

# Clean Intersection_Type
crash_more_grouping_df["Intersection_Type"] = crash_more_grouping_df["Intersection_Type"].replace(
    ["ROUNDABOUT", "TRAFFIC CIRCLE"], "Roundabout/Traffic Circle"
)

# Convert all values to uppercase
crash_more_grouping_df["Intersection_Type"] = crash_more_grouping_df["Intersection_Type"].astype(str).str.upper()

# Convert the string 'NAN' back to actual np.nan
crash_more_grouping_df["Intersection_Type"] = crash_more_grouping_df["Intersection_Type"].replace("NAN", np.nan)

print("\nValue counts for Intersection_Type:")
print(crash_more_grouping_df["Intersection_Type"].value_counts(dropna=False))


Value counts for Intersection_Type:
Intersection_Type
NaN                          112283
FOUR-WAY INTERSECTION         60999
T-INTERSECTION                26096
PERPENDICULAR                 10258
ANGLED/SKEWED                  2353
Y-INTERSECTION                 1160
ROUNDABOUT/TRAFFIC CIRCLE       734
FIVE-POINT OR MORE              390
Name: count, dtype: int64


In [87]:
#  save cleaned df to csv to back up

crash_more_grouping_df.to_csv("drivers_data_final_df.csv", index=False)

# index = False prevents "Unnamed: 0" column from being added each time file loads.


Drivers_License_State  change
"XX" to nan,
 change "MX-MEX", "MX-MX-ROO", "MX-GRO" to "Mex",
change CA-ON, CA-QC, ON, QC, BC, and any other codes that are canadian provinces to "Can"

To be cleaned up now:  

- "Intersection_Type"  

- Lane_Type to Parking_Lot Y/N/nan  
check against off road

- Vehicle_Damage:  caseblind, "unkonwn" = "other" = nan

Veh_1st_Impact_Loc : "ROOF TOP" = "roof" = "top"; caseblind; OCLOCK = O Clock to O'Clock

- Driver_Distraction:  Unknown = UNKNOWN = "NO DRIVER PRESENT" nan

- "Collision_Type"  "other" = "unknown" = nan

- Veh_Body_Type : caseblind,  unknown = other,

- First_Harmful_Event :   
- Second_Harmful_Event :  

- Junction : "Non-Junction" =  "NON INTERSECTION" = "UNKNOWN"  = nan

- Intersection_Type :   
"Roundabout/Traffic Circle" = "ROUNDABOUT" = "TRAFFIC CIRCLE"

- Road Alignment: if not Straight or nan then change to curved



####Veh_Body_Type Notes, Explanations
"PASSENGER CAR" includes:
- Sedan, hatchback
- STATION WAGON (949)


"OTHER" includes:
- MOTORCYCLE - 3 WHEELED (3)
- LIMOUSINE (16)
- LOW SPEED VEHICLE (40)
- GOLF CART (3)
- AUTOCYCLE (50)
- RECREATIONAL OFF-HIGHWAY VEHICLES (ROV) (7)  
- FARM EQUIPMENT (TRACTOR, COMBINE HARVESTER, ETC.) (3)
- FARM VEHICLE (21)
- CONSTRUCTION EQUIPMENT (BACKHOE, BULLDOZER, ETC.) (25)
- SNOWMOBILE (130)

"VAN" assume Passenger (because otherwise specified)

####Note:  
For "CARGO VAN/LIGHT TRUCK 2 AXLES (OVER 10,000LBS (4,536 KG))",  
the weight is likely to be a mistake in the  
Automated Crash Reporting System (ACRS).

###Buses:
Buses are not grouped, because school bus circumstances are very  
different from local transit buses.  
Both are different from long-distance coaches, which are mostly on highways.

####Groupings for later:
Can group emergency vehicles in different ways.  
For now, want to explore the differences.

####Categorical Data:
ACRS_Report_Type : ['Injury Crash' 'Property Damage Crash' 'Fatal Crash']



####Date/Time:
Date_Time

####Clock Positions and Crash Impact
Twelve o'clock = front bumper  
Three  o'clock = right (US passenger) side center  
Six o'clock = rear bumper  
Nine o'clock = left (US driver) side center  

Alaska Department of Public Safety  
https://tracs.dps.alaska.gov/TechSupport/12-200.1/MotorVehicle/DamagedAreas.htm

------
####Driver_Distraction  

these categories are good.  
Optional to later group into  
Device-related,  
Passenger-related  
Vehicle device controls  
etc.


Collision_Type, Driver_Distraction remain,
although probably won't use for current analysis.

---------------
####Future research:  
It would be useful to interview representatives of the municipalities about how they classify and record specific details about crashes, such as drug and alcohol use.  
I would also like to know why the handbook lists many more fields in covered by the Automated Crash Reporting System (ACRS)

Maryland Department of State Police
https://mdsp.maryland.gov/Pages/Dashboards/CrashDataDashboard.aspx  
Maryland Crash Data 2024-Present
Map  
Uses updated information, but that information is not shared with the public (only the preliminary information, which may be incomplete or contain errors for the reasons listed).

Age of driver

Include data about passengers:  number of passengers, general age
------------

####Note on Data
In trying to understand the meaning of certain fields, I may have found an error in the Incidents data description on  
https://data.montgomerycountymd.gov/Public-Safety/Crash-Reporting-Incidents-Data/bhju-22kf/about_data  
The "Direction" field is described as "Location - Direction from mile point."  
The "Distance" field is described as "Location - Distance from mile point."  

However, the Maryland Automated Crash Reporting System (ACRS) Field Reference Guide published on  
https://www.nhtsa.gov/sites/nhtsa.gov/files/documents/acrsfieldreference.pdf
(2017 Maryland State Police)  
section 2.19 states:  
"Distance
Definition:
The distance from the referenced Intersecting Road to the crash site.
Explanation:
The distance in feet or miles from the Intersecting Road to crash site."

Section 2.21:  
"Distance Direction
Definition:
The compass direction describing the direction going from the primary and intersecting
roads to the crash.
Explanation:
Units are described in compass points, i.e. North, South East and West. If the crash
occurred in the intersection, select the compass point referenced in the Mile Point
Direction box above."
---------------------


####About Data

Crash Reporting - Incidents Data  
Original data:  
122,000 Rows  
37 Columns  
Each row is a "Collision"

https://data.montgomerycountymd.gov/Public-Safety/Crash-Reporting-Incidents-Data/bhju-22kf/about_data  

"This dataset provides general information about each collision and details of all traffic collisions occurring on county and local roadways within Montgomery County, as collected via the Automated Crash Reporting System (ACRS) of the Maryland State Police, and reported by the Montgomery County Police, Gaithersburg Police, Rockville Police, or the Maryland-National Capital Park Police.

Please note that these collision reports are based on preliminary information supplied to the Police Department by the reporting parties. Therefore, the collision data available on this web page may reflect:

-Information not yet verified by further investigation  
-Information that may include verified and unverified collision data  
-Preliminary collision classifications may be changed at a later date based   upon further investigation  
-Information may include mechanical or human error"

"3.38 Going Direction
Definition:
The direction of the motor vehicles travel on the roadway prior to the crash.
Explanation:
This is not necessarily a compass direction, but must be consistent with the "Distance
Direction" from the Log Mile Information for this roadway."


![crash map disclaimer](https://raw.githubusercontent.com/Alissa-Ouspen/data201_alissa/main/final/crash_map_disclaimer.png)




####Sources:

- Learn types of intersections | Complete Guide to 12 Types  
https://www.dmvpermittests.com/blog/different-types-of-intersections

IGNORE

AI usage tracker:

- For all cells in merged_cleaned_df:
if value in cell == "unkonwn" or "UNKNOWN" or "other" or "OTHER" or "Not Applicable" or "NOT APPLICABLE",  
set value to nan  

- Make all caseblind:  
Compare values in each column.  
If a value written in all caps has a matching value that is written in mixed case or lower case,  
then convert all data written in all caps into the matching version that is written in mixed case or lower case.

- Drivers_License_State change "XX" to nan, change "MX-__" to "Mex", change CA-ON, CA-QC, ON, QC, BC, and any other codes that are canadian provinces to "Can".  List all US states and DC.  List all US territories and create Territories group. Create Foreign/Other group for all else except nan.  

- Veh_1st_Impact_Loc
"ROOF TOP", "roof" : "top"   
OCLOCK, O Clock : O'Clock

- First_Harmful_Event:  
analyze text in unique values and suggest values that seem similar or very related. Do not write code to change the grouping at this time. Only suggest values that could potentially be combined into larger groups for the purpose of ending up with fewer categories
